### Importing packages and modules

In [1]:
# module for building the pyomo model
import pyomo.environ as pe
# module for solving the pyomo model
import pyomo.opt as po

from itertools import product

### Create the model

In [2]:
model = pe.ConcreteModel()

Order to build the model:
1. Sets
1. Parameters
1. Variables
1. Objective function
1. Constraints

#### Sets

$d$: demands {'d1', 'd2'}

In [3]:
model.demands = pe.Set(initialize=['d1', 'd2'])

$g$: generators {'g1', 'g2'}

In [4]:
model.generators = pe.Set(initialize=['g1', 'g2'])

$n$: nodes {'n1', 'n2', 'n3'}

In [5]:
model.nodes = pe.Set(initialize=['n1', 'n2', 'n3'])

#### Parameters

$U_{d}$: bid price of demand d

In [6]:
bid_price_demand = {
    'd1': 40,
    'd2': 35
}

model.bid_price_demand = pe.Param(model.demands, initialize = bid_price_demand)

$C_{g}$: offer price of generator g

In [7]:
offer_price_generator = {
    'g1': 12,
    'g2': 20
}

model.offer_price_generator = pe.Param(model.generators, initialize = offer_price_generator)

$\overline{P_d^D}$: maximum load of demand d

In [8]:
max_load_demand = {
    'd1': 100,
    'd2': 50
}

model.max_load_demand = pe.Param(model.demands, initialize = max_load_demand)

$\overline{P_g^G}$: capacity of generator g

In [9]:
capacity_generator = {
    'g1': 100,
    'g2': 80
}

model.capacity_generator = pe.Param(model.generators, initialize = capacity_generator)

$DCN_{d, n}$: is demand d connected to node n

In [10]:
demand_connection = {
    ('d1', 'n2'): 1,
    ('d2', 'n3'): 1
}

model.demand_connection = pe.Param(model.demands, model.nodes, initialize = demand_connection, default=0)

$GCN_{g, n}$: is generator g connected to node n

In [11]:
generator_connection = {
    ('g1', 'n1'): 1,
    ('g2', 'n2'): 1
}

model.generator_connection = pe.Param(model.generators, model.nodes, initialize = generator_connection, default=0)

$NCN_{n, m}$: is node n connected to node m

In [12]:
node_connection = {
    ('n1', 'n2'): 1,
    ('n2', 'n1'): 1,
    
    ('n2', 'n3'): 1,
    ('n3', 'n2'): 1,
     
    ('n1', 'n3'): 1,
    ('n3', 'n1'): 1,
}

model.node_connection = pe.Param(model.nodes, model.nodes, initialize = node_connection, default=0)

$B_{n, m}$: susceptance from node n to node m

In [13]:
node_susceptance = {
    ('n1', 'n2'): 500,
    ('n2', 'n1'): 500,
    
    ('n2', 'n3'): 500,
    ('n3', 'n2'): 500,
    
    ('n1', 'n3'): 500,
    ('n3', 'n1'): 500,
}

model.node_susceptance = pe.Param(model.nodes, model.nodes, initialize = node_susceptance, default=500)

$F_{n, m}$: maximum power flow from node n to node m

In [14]:
maximum_power_flow = {
    ('n1', 'n2'): 100,
    ('n2', 'n1'): 100,
    
    ('n2', 'n3'): 100,
    ('n3', 'n2'): 100,
    
    ('n1', 'n3'): 100,
    ('n3', 'n1'): 100,
}

model.maximum_power_flow = pe.Param(model.nodes, model.nodes, initialize = maximum_power_flow, default=100)

#### Variables

$p_d^D$: quantity purchased by demand d

In [15]:
model.quantity_purchased_demand = pe.Var(model.demands, within = pe.NonNegativeReals)

$p_g^G$: quantity purchased to generator g

In [16]:
model.quantity_purchased_generator = pe.Var(model.generators, within = pe.NonNegativeReals)

$\theta_n$: voltage angle of node n

In [17]:
model.voltage_angle = pe.Var(model.nodes, within = pe.Reals)

#### Objective Function

max $\sum_{d} U_{d} \space p_d^D - \sum_{g} C_{g} \space p_g^G$

In [18]:
def obj_rule(model):
    return sum(model.bid_price_demand[d] * model.quantity_purchased_demand[d] for d in model.demands) - sum(model.offer_price_generator[g] * model.quantity_purchased_generator[g] for g in model.generators)

model.cost = pe.Objective(rule = obj_rule, sense = pe.maximize)

#### Constraints

Constraint #1: maximum load of demand d

$ p_d^D \leq \overline{P_d^D} \quad \forall d$

In [19]:
model.constraint_demand_max_load = pe.ConstraintList()

for d in model.demands:
    model.constraint_demand_max_load.add(
        model.quantity_purchased_demand[d] <= model.max_load_demand[d]
    )

Constraint #2: maximum capacity of generator g

$ p_g^G \leq \overline{P_g^G} \quad \forall d$

In [20]:
model.constraint_generator_capacity = pe.ConstraintList()

for g in model.generators:
    model.constraint_generator_capacity.add(
        model.quantity_purchased_generator[g] <= model.capacity_generator[g]
    )

Constraint #3: ensure balance in demand, production and transportation in nodes

$ \sum_{m} B_{n, m} \space (\theta_n - \theta_m) \space NCN_{n, m} + \sum_{d} p_d^D \space DCN_{d, n} - \sum_{g} p_g^G \space GCN_{g, n} = 0 \quad \forall n$

In [21]:
def node_balance_rule(model, n):
    return (sum(model.node_susceptance[n, m]*(model.voltage_angle[n] - model.voltage_angle[m])*model.node_connection[n, m] 
               for m in model.nodes) + 
            sum(model.quantity_purchased_demand[d]*model.demand_connection[d, n] 
                for d in model.demands) - 
            sum(model.quantity_purchased_generator[g]*model.generator_connection[g, n] 
                for g in model.generators)) == 0

model.constraint_node_balance = pe.Constraint(model.nodes, rule=node_balance_rule)

Constraint #4: maximum flow between nodes

$ -F_{n, m} \leq B_{n, m} \space (\theta_n - \theta_m) \space NCN_{n, m} \leq F_{n, m} \quad \forall n, m$

In [22]:
model.constraint_max_flow = pe.ConstraintList()

for n in model.nodes:
    for m in model.nodes:
        model.constraint_max_flow.add(
            model.node_susceptance[n, m]*(model.voltage_angle[n] - model.voltage_angle[m])*model.node_connection[n, m] <= model.maximum_power_flow[n, m]
        )
        
        model.constraint_max_flow.add(
            -model.maximum_power_flow[n, m] <= model.node_susceptance[n, m]*(model.voltage_angle[n] - model.voltage_angle[m])*model.node_connection[n, m]
        )

Constraint #5: voltage angle ref equals 0

$ \theta_{n1} = 0$

In [23]:
def ref_rule(model):
    return model.voltage_angle[model.nodes.at(1)] == 0

model.constraint_ref = pe.Constraint(rule=ref_rule)

#### Solver definition and solve statement

In [ ]:
solver = po.SolverFactory('gurobi')

if not hasattr(model, 'dual'):
	model.dual = pe.Suffix(direction=pe.Suffix.IMPORT)

results = solver.solve(model, tee=True)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-03
Read LP format model from file C:\Users\avela\AppData\Local\Temp\tmptyyseo70.pyomo.lp
Reading time = 0.00 seconds
x1: 26 rows, 8 columns, 42 nonzeros
Set parameter QCPDual to value 1
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (win64 - Windows 11+.0 (26100.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 26 rows, 8 columns and 42 nonzeros
Model fingerprint: 0xd7732fd3
Coefficient statistics:
  Matrix range     [1e+00, 1e+03]
  Objective range  [1e+01, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+01, 1e+02]
Presolve removed 25 rows and 4 columns
Presolve time: 0.00s
Presolved: 1 rows, 4 columns, 4 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    5.7500000e+03   1.875000e+01   0.000000e+0

In [ ]:
for n in model.nodes:
    dual_val = model.dual.get(model.constraint_node_balance[n])
    if dual_val is not None:
        print(f"Dual value of balance constraint for node {n}: {dual_val}")
    else:
        print(f"Dual value of balance constraint for node {n} not available")

data_to_show_demands = {
	"names": ["$U_d$", "$p_d^D$", "$\overline{P_d^D}$"],
	"values": [model.bid_price_demand, model.quantity_purchased_demand, model.max_load_demand],
	"elements": [model.demands]
}

data_to_show_generators = {
	"names": ["$C_g$", "$p_g^G$", "$\overline{P_g^G}$"],
	"values": [model.offer_price_generator, model.quantity_purchased_generator, model.capacity_generator],
	"elements": [model.generators]
}

def difference_angle_voltages(args):
	return f"{pe.value(model.node_susceptance[args])*(pe.value(model.voltage_angle[args[0]]) - pe.value(model.voltage_angle[args[1]]))}"

data_to_show_connections = {
	"names": ["$\\theta_n - \\theta_m$", "$F_{n, m}$"],
	"values": [difference_angle_voltages, model.maximum_power_flow],
	"elements": [model.nodes, model.nodes]
}

def correct_length(text, length:int, filler=" ", end_extra_fill:bool=True, start_extra_fill:bool=True):
    return f"{filler if start_extra_fill else ''}{filler*(length - len(str(text)))}{text}{filler if end_extra_fill else ''}"

def create_matrix(data_to_show):
	if len(data_to_show["names"]) != len(data_to_show["values"]):
		print("Length of data_to_show['names'] and data_to_show['values'] should be equal.")
		raise ValueError
	matrix = []
	matrix.append(["", *data_to_show["names"]])
	matrix.append(["" for _ in range(len(data_to_show["names"]) + 1)])
	combinations = list(product(*data_to_show["elements"]))
	for combination in combinations:
		matrix.append([str(combination), *[str(data(combination) if str(type(data)) == "<class 'function'>" else pe.value(data[combination])) for data in data_to_show["values"]]])
	for j in range(len(matrix[0])):
		# Get max legth of column
		max_length = max(len(matrix[i][j]) for i in range(len(matrix)))
 
		# Fill column elements to max length
		for i in range(len(matrix)):
			if i == 1:
				matrix[i][j] = correct_length(matrix[i][j], max_length, filler="-")
			else:
				matrix[i][j] = correct_length(matrix[i][j], max_length)

	return matrix

def show_table(data_to_show):
	matrix = create_matrix(data_to_show)
	print()
	print(" " + "|".join(matrix[0]) + "|")
	print(" " +  " ".join(matrix[1]))
	for i in range(2, len(matrix)):
		print("|" + "|".join(matrix[i]) + "|")

show_table(data_to_show_demands)
show_table(data_to_show_generators)
show_table(data_to_show_connections)

Dual value of balance constraint for node n1: 20.0
Dual value of balance constraint for node n2: 20.0
Dual value of balance constraint for node n3: 20.0

          | $U_d$ | $p_d^D$ | $\overline{P_d^D}$ |
 --------- ------- --------- --------------------
| ('d1',) |    40 |   100.0 |                100 |
| ('d2',) |    35 |    50.0 |                 50 |

          | $C_g$ | $p_g^G$ | $\overline{P_g^G}$ |
 --------- ------- --------- --------------------
| ('g1',) |    12 |   100.0 |                100 |
| ('g2',) |    20 |    50.0 |                 80 |

               | $\theta_n - \theta_m$ | $F_{n, m}$ |
 -------------- ----------------------- ------------
| ('n1', 'n1') |                   0.0 |        100 |
| ('n1', 'n2') |                  50.0 |        100 |
| ('n1', 'n3') |                  50.0 |        100 |
| ('n2', 'n1') |                 -50.0 |        100 |
| ('n2', 'n2') |                   0.0 |        100 |
| ('n2', 'n3') |                   0.0 |        100 |
| ('n3'